# IEEE 14-bus AC OPF — forecast error and hidden congestion

**Thursday mini-project:** connect Day-1 cvxpy dispatch duals to grid physics with pandapower.

- Load IEEE 14-bus, run power flow, then AC OPF with generator costs
- Extract **bus LMPs** (power-balance shadow prices) and **line loadings**
- Perturb loads with a synthetic day-ahead load forecast vs actual profile
- Show that forecast error is not just an accuracy miss — it shifts prices and can hide congestion

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.dispatch.ieee14_opf import (
    BINDING_LOADING_THRESHOLD,
    compare_binding_status,
    load_ieee14_case,
    prepare_congestion_case,
    run_power_flow,
    solve_opf_at_scale,
)
from src.eval.opf_scenario import run_opf_scenarios
from src.forecast.load_forecaster import (
    build_system_load_profile,
    forecast_vs_actual_scales,
    load_forecast_mae,
    stress_actual_scales,
)

RESULTS = Path("../results")
RESULTS.mkdir(exist_ok=True)
%matplotlib inline

## 1. Baseline AC OPF on IEEE 14-bus

Day 1 used a single aggregate power-balance constraint in cvxpy. Its dual is one scalar LMP per hour.

AC OPF on a network gives **locational** LMPs — the shadow price of serving one more MW at each bus. When a transmission limit binds, LMPs diverge across buses (congestion component).

In [ ]:
net = load_ieee14_case()
prepare_congestion_case(net)
assert run_power_flow(net)

baseline = solve_opf_at_scale(net, scale=1.0)
print(f"OPF converged: {baseline.converged}, total cost ${baseline.total_cost:,.0f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
baseline.bus_lmps.sort_index().plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Bus LMPs ($/MWh)")
axes[0].set_xlabel("Bus")
axes[0].set_ylabel("LMP")
axes[0].grid(True, alpha=0.3)

top = baseline.line_loadings["loading_percent"].sort_values(ascending=False).head(8)
top.plot(kind="barh", ax=axes[1], color="coral")
axes[1].set_title("Top line loadings (%)")
axes[1].set_xlabel("Loading %")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(RESULTS / "ieee14_lmp.png", dpi=150)
plt.show()

## 2. Load forecast vs actual — scaling bus loads

We reuse the Week 1/4 pattern: lag + calendar features on a synthetic system load profile (Day-1 sinusoid + noise). Forecast and actual **scale factors** multiply every bus load uniformly.

In [ ]:
profile = build_system_load_profile(days=30, seed=42)
eval_day = profile.index.normalize().unique()[-1]
actual, forecast = forecast_vs_actual_scales(profile, eval_day)
actual_stressed = stress_actual_scales(actual, forecast)

hours = pd.RangeIndex(0, 24, name="hour")
pd.DataFrame({
    "actual": actual,
    "forecast": forecast,
    "actual_opf": actual_stressed,
}, index=hours).plot(figsize=(10, 4), title=f"Load scale factors — eval day {eval_day.date()}")
plt.ylabel("System load scale")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Load-scale MAE: {load_forecast_mae(actual, forecast):.3f}")

## 3. Hidden congestion under forecast error

Tighten the heaviest corridor (line 0, buses 0–1). Run OPF under **forecast** load scale vs **actual** (stressed when forecast under-predicts peak).

**Hidden congestion:** a line binds in the actual-load case but not in the forecast-load case — the operator's model sees no binding constraint.

In [ ]:
demo_net = load_ieee14_case()
prepare_congestion_case(demo_net, cap_mult=0.87)

peak_hour = int(actual.argmax())
fc = solve_opf_at_scale(demo_net, scale=float(forecast[peak_hour]))
ac = solve_opf_at_scale(demo_net, scale=float(stress_actual_scales(actual, forecast)[peak_hour]))
binding = compare_binding_status(fc, ac, threshold=BINDING_LOADING_THRESHOLD)
display(binding.sort_values("actual_loading_pct", ascending=False).head())

hidden = binding[binding["hidden_congestion"]]
print(f"Peak hour {peak_hour}: hidden congestion lines = {hidden['line'].tolist()}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
lmp_shift = (ac.bus_lmps - fc.bus_lmps).sort_index()
lmp_shift.plot(kind="bar", ax=axes[0], color="purple")
axes[0].axhline(0, color="k", lw=0.8)
axes[0].set_title(f"LMP shift (actual − forecast) at hour {peak_hour}")
axes[0].set_ylabel("$/MWh")
axes[0].grid(True, alpha=0.3)

cmp = binding[["forecast_loading_pct", "actual_loading_pct"]].set_index(binding["line"])
cmp.loc[cmp.max(axis=1).sort_values(ascending=False).head(6).index].plot(
    kind="barh", ax=axes[1]
)
axes[1].axvline(BINDING_LOADING_THRESHOLD, color="r", ls="--", label="binding threshold")
axes[1].set_xlabel("Loading %")
axes[1].set_title("Line loading: forecast vs actual")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(RESULTS / "ieee14_scenario_comparison.png", dpi=150)
plt.show()

## 4. Physical story

A load forecast that looks acceptable on MAE can still be **dangerous for operations**:

1. **Prices move.** Bus LMPs shift when realized load differs from the OPF input — generators and storage see the wrong price signal.
2. **Congestion can be invisible.** If forecast load keeps a line below its limit in the modeled case, the operator does not see binding constraints — but actual load can bind the same line, creating local price spikes and reliability risk.
3. **Same dual object as Day 1.** Scalar hourly LMPs from cvxpy and bus LMPs from pandapower are both shadow prices on power balance — the network adds **where** congestion splits the price.

This is why the project tracks **decision regret**, not forecast MAE alone: the optimization layer converts errors into dollars and grid stress.

Run the full 24-hour scenario CSV with:

```powershell
python -m src.eval.opf_scenario --hours 24
```